# 02_preprocess_unsup.ipynb
## Purpose
Normalize/clean segments for the UNSUP branch (e.g., lowercasing, spacing, noise tokens).

## Expected inputs
- `data/segments_index.csv`

## Expected outputs
- `intermediate processed text columns used by UNSUP ranking`

## Notes
- Keys are aligned using `seg_key = comment_id__seg_id`.

# Preprocessing Unsupervised for `seg_text`
- Default input: column **`seg_text`** (results segmentation).
- Handle **emoticons/emoji** correctly:
- Unicode emoji (😂🤣🙏👍)
- ASCII emoticons (:) :-D ;-P)
- Forum/Kaskus-style `:ngakak` and `:ngakak:` (remove completely, not convert to words).
- Handle **repeated 2-digit words**: `negara2 → negara-negara` (safe for parsing and topic modeling).
- **Custom stopwords**: can be loaded from file (1 stopword per row).
Two stopword modes are available:
- `UNSUP` (more aggressive for topic modeling)
- `ABSA_SAFE` (does not remove negations such as *not/gak/no/not*)
- Output:
- `unsup_processed` (clean + slang)
- `unsup_filtered` (stopword unsup)
- `unsup_stemmed` (optional)
- `unsup_filtered2` (filtered after stemming)
- `absa_filtered` (sentiment-safe/rule-based stopwords)

Additionally, an audit is created:
- `audit_preprocess_unsup.json`
- `audit_samples_unsup.csv`

> Minimal column: `seg_text`. Jika beda, change `TEXT_COL`.

In [ ]:
# =========================
# 0) Setup
# =========================
import re, json
from dataclasses import dataclass
from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd

print("Ready.")

## 1) I/O configuration

In [ ]:
# =========================
# 1) I/O
# =========================
INPUT_CSV = Path("data/dataset.csv")
TEXT_COL  = "seg_text"          # <-- default untuk segmen

# Kamus slang format: slang:formal (optional)
SLANG_FILE = Path("data/slangword.txt")

# Custom stopwords (1 toton per row) - optional
CUSTOM_STOPWORDS_UNSUP = Path("data/custom_stopwords.txt")   # lebih agresif
# CUSTOM_STOPWORDS_ABSA  = Path("custom_stopwords_absa.txt")    # aman for sentimen (tanpa negasi)
CUSTOM_STOPWORDS_ABSA  = None   # aman untuk sentimen (tanpa negasi)

OUT_CSV = Path("data/preprocessedfor_unsup.csv")
AUDIT_JSON = Path("data/audit_preprocess_unsup.json")
AUDIT_SAMPLES = Path("data/audit_samples_prep_unsup.csv")

INPUT_CSV, OUT_CSV

# ===

## 2) Loader utility: slang, stopwords, stemmer

In [ ]:
def _safe_read_text(path: Path, encoding: str = "utf-8") -> str:
    return path.read_text(encoding=encoding, errors="ignore")


def load_slang_map(path: Optional[Path]) -> Dict[str, str]:
    slang = {}
    if not path or not Path(path).exists():
        print(f"[WARN] slang file not found: {path}")
        return slang
    raw = _safe_read_text(Path(path))
    for line in raw.splitlines():
        line = line.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        k, v = line.split(":", 1)
        k = k.strip().lower()
        v = v.strip().lower()
        if k and v:
            slang[k] = v
    return slang


def load_wordlist(path: Optional[Path]) -> List[str]:
    if not path or not Path(path).exists():
        print(f"[WARN] wordlist file not found: {path}")
        return []
    raw = _safe_read_text(Path(path))
    out = []
    for line in raw.splitlines():
        w = line.strip()
        if not w or w.startswith("#"):
            continue
        out.append(w)
    return out


def try_load_stopwords() -> Tuple[set, set]:
    try:
        import nltk  # noqa
        from nltk.corpus import stopwords
        try:
            return set(stopwords.words("indonesian")), set(stopwords.words("english"))
        except LookupError:
            print("[WARN] NLTK stopwords not downloaded. Run: nltk.download('stopwords')")
            return set(), set()
    except Exception:
        return set(), set()


def try_load_stemmer():
    try:
        from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
        return StemmerFactory().create_stemmer()
    except Exception:
        return None


slang_map = load_slang_map(SLANG_FILE)
indo_sw, eng_sw = try_load_stopwords()
stemmer = None  # stemming disabled (removed as requested)
stop_unsup_custom = set([w.strip().lower() for w in load_wordlist(CUSTOM_STOPWORDS_UNSUP)])
stop_absa_custom  = set([w.strip().lower() for w in load_wordlist(CUSTOM_STOPWORDS_ABSA)])

print("slang_map:", len(slang_map))
print("stopwords NLTK indo:", len(indo_sw), "eng:", len(eng_sw))
print("custom stop UNSUP:", len(stop_unsup_custom), "custom stop ABSA:", len(stop_absa_custom))
print("Sastrawi:", "DISABLED")

## 3) Regex & Preprocessing Configuration


In [ ]:
# =========================
# 3) Regex precompiled
# =========================
# _URL_RE = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
_URL_RE = re.compile(
    r"(?i)\b(?:https?://|www\.)[^\s<>()\[\]{}\"']+"
)
# --- Placeholder URL such as <URL>, <url>, < Url >
URL_TOKEN_RE = re.compile(r"(?i)<\s*url\s*>")

# --- Variasi dash: –, — (en/em dash)
DASH_CHARS_RE = re.compile(r"[–—]")

# --- Hapus toton dash beruntun yang berdiri sendiri: "--", "---", "——", dsb
# (not mengganggu word ber-hyphen tunggal: "base-load" tetap aman)
DASH_RUN_RE = re.compile(r"(?<!\w)[-–—]{2,}(?!\w)")

_MENTION_RE = re.compile(r"@[A-Za-z0-9_]+")
_RT_RE = re.compile(r"\bRT\b", flags=re.IGNORECASE)
_HASHTAG_RE = re.compile(r"#([A-Za-z0-9_]+)")

_WHITESPACE_RE = re.compile(r"\s+")
# angka murni (bukan alfanum nempel)
_NUM_ONLY_RE = re.compile(r"\b\d+(?:[\.,]\d+)?\b")

# Emoticon/emoji
KASKUS_EMO_RE = re.compile(
    r"(?<!\w):[A-Za-z][\w\-]{1,30}:(?!\w)|(?<!\w):[A-Za-z][\w\-]{1,30}(?!\w)"
)
ASCII_EMO_RE = re.compile(r"(?::|;|=)(?:-)?(?:\)|\(|D|P|p|/|\\)")
EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA70-\U0001FAFF"
    "\u2600-\u26FF"
    "\u2700-\u27BF"
    "]+"
)
ZWJ_VS_RE = re.compile(r"[\u200d\uFE0F]")

# Redup2: negara2 -> negara-negara (hinfrom CO2/H2O/G20)
EXC_REDUP2 = {"co2", "h2o", "g20"}
REDUP2_RE = re.compile(r"\b([a-z]{3,})2\b", flags=re.IGNORECASE)

# Whitelist for unsupervised: huruf+digit+spasi+hyphen+<NUM>
# _ALLOWED_UNSUP_RE = re.compile(r"[^0-9A-Za-zÀ-ÖØ-öø-ÿ\s\-\<\>]+")

_ALLOWED_UNSUP_RE = re.compile(r"[^0-9A-Za-zÀ-ÖØ-öø-ÿ\s]+")

# punctuation remover (kita buang semua tocuali hyphen)
_PUNCT_RE = re.compile(r"[^\w\s\-<>]+")


# =========================
# 3b) Energy orthography normalization (WHITELIST) + join rules
#     - Minimal addition: does NOT change CFG / pipeline structure
# =========================
JOIN_RULES_2 = {
    ("batu","bara"): "batubara",
    ("panas","bumi"): "panas_bumi",          # geothermal
    ("gas","bumi"): "gas_bumi",
    ("minyak","bumi"): "minyak_bumi",
    ("minyak","mentah"): "minyak_mentah",
    ("tagihan","listrik") : "tagihan_listrik",
    ("tarif","listrik"): "tarif_listrik",
    ("harga", "listrik"): "harga_listrik",
    ("biaya", "listrik"): "biaya_listrik",
    ("harga","energi"):"harga_energi",
    ("energi","terbarukan"): "energi_terbarukan",
    ("bahan","bakar"): "bahan_bakar",
    ("carbon","capture"): "carboncapture",
    ("jaringan", "listrik"): "jaringan_listrik",
    ("kendaraan", "listrik"): "kendaraan_listrik",
    ("mobil", "listrik"):"mobil_listrik",
    ("token","listrik"):"token_listrik",
    ("byar","pet"):"pemadaman",
    ("black","out"):"pemadaman",
    ("kapasitas","daya"):"daya",
    ("pembangkit","uap"):"pltu",
    ("pembangkit","gas"):"pltg",
    ("mini","hidro"):"pltm",
    ("mini","hydro"):"pltm",
    ("makro","hidro"):"pltm",
    ("macro","hydro"):"pltm",
    ("transisi", "energi"):"transisi_energi",
    ("green","energy"):"energi_hijau",
    ("ramah","lingkungan"):"ramah_lingkungan",
    ("net","zero"):"net_zero",
    ("polusi","udara"):"polusi_udara",
    ("base","load"):"base_load",
    ("reserve","margin"): "reserve_margin",
    ("cadangan","daya"): "cadangan_daya",
    ("kapasitas", "cadangan"):"kapasitas_cadangan",
    # optional but often used
}

# Regex whitelist for variasi spasi/hyphen/underscore
_ENERGY_VARIANTS = [
    ("batubara", r"\bbatu[\s\-_]*bara\b"),
    ("panas_bumi", r"\bpanas[\s\-_]*bumi\b"),
    ("gas_bumi", r"\bgas[\s\-_]*bumi\b"),
    ("minyak_bumi", r"\bminyak[\s\-_]*bumi\b"),
    ("energi_terbarukan", r"\benergi[\s\-_]*terbarukan\b"),
    ("carboncapture", r"\bcarbon[\s\-_]*capture\b"),
]

# Tambahkan pola turunan from JOIN_RULES_2 agar hyphen/underscore juga tersatukan
_DERIVED_VARIANTS = []
for (a, b), canon in JOIN_RULES_2.items():
    _DERIVED_VARIANTS.append((canon, rf"\b{re.escape(a)}[\s\-_]+{re.escape(b)}\b"))

# compile (lebih dulu yang long supaya not topotong)
_ENERGY_REGEXES = [(c, re.compile(p)) for (c, p) in (_ENERGY_VARIANTS + _DERIVED_VARIANTS)]
_ENERGY_REGEXES.sort(key=lambda cp: len(cp[1].pattern), reverse=True)

def normalize_energy_variants(text: str) -> str:
    if not text:
        return text
    t = text
    # asumsi t sudah lowercase di pipeline
    for canon, pat in _ENERGY_REGEXES:
        t = pat.sub(canon, t)
    return t

def apply_join_rules_text(text: str) -> str:
    if not text:
        return text
    toks = text.split()
    out = []
    i = 0
    while i < len(toks):
        if i < len(toks) - 1:
            key = (toks[i], toks[i+1])
            if key in JOIN_RULES_2:
                out.append(JOIN_RULES_2[key])
                i += 2
                continue
        out.append(toks[i])
        i += 1
    return " ".join(out)

# Untuk audit: patterns before (raw) vs canonical setelah (processed)
ENERGY_PATTERNS_FOR_AUDIT = [(canon, pat.pattern) for canon, pat in _ENERGY_REGEXES]
@dataclass
class UnsupConfig:
    lowercase: bool = True
    keep_hashtag_words: bool = True
    keep_numbers: str = "token"  # token|keep|remove
    redup2_style: str = "base" # hyphen|space|base (base = kata2 -> kata)
    remove_punct: bool = True


CFG = UnsupConfig()
CFG

## 4) Main Function: clean_unsup + slang normalize + stopword removal

In [ ]:
def normalize_slang(text: str, slang_map: Dict[str, str]) -> str:
    if not text or not slang_map:
        return text
    t = text

    # multiword first
    multi = [(k, v) for k, v in slang_map.items() if " " in k]
    multi.sort(key=lambda kv: len(kv[0]), reverse=True)
    for k, v in multi:
        t = re.sub(r"(?<!\w)" + re.escape(k) + r"(?!\w)", v, t)

    # single word
    for k, v in ((k, v) for k, v in slang_map.items() if " " not in k):
        t = re.sub(r"\b" + re.escape(k) + r"\b", v, t)

    return t


def _apply_redup2(t: str, cfg: UnsupConfig) -> str:
    def _repl(m):
        w = m.group(1)
        if (w + "2").lower() in EXC_REDUP2:
            return w + "2"
        if cfg.redup2_style == "hyphen":
            return f"{w}-{w}"
        if cfg.redup2_style == "space":
            return f"{w} {w}"
        if cfg.redup2_style == "base":
            return w
        return f"{w}-{w}"
    return REDUP2_RE.sub(_repl, t)


def clean_unsup(text: str, cfg: UnsupConfig) -> str:
    if text is None:
        return ""
    t = str(text)

    if cfg.lowercase:
        t = t.lower()

    # ==
    # 1) remove URL asli and toton <URL>
    t = _URL_RE.sub(" ", t)
    t = URL_TOKEN_RE.sub(" ", t)

    # 2) normalisasi en/em dash jadi "-" supaya konsisten
    t = DASH_CHARS_RE.sub("-", t)

    # ===

    # remove url/mention/rt
    # t = _URL_RE.sub(" ", t)
    t = _MENTION_RE.sub(" ", t)
    t = _RT_RE.sub(" ", t)

    # hashtags
    if cfg.keep_hashtag_words:
        t = _HASHTAG_RE.sub(r"\1", t)
    else:
        t = _HASHTAG_RE.sub(" ", t)

    t = t.replace("\n", " ").replace("\r", " ")

    # remove emoticons/emoji EARLY
    t = KASKUS_EMO_RE.sub(" ", t)
    t = ASCII_EMO_RE.sub(" ", t)
    t = EMOJI_RE.sub(" ", t)
    t = ZWJ_VS_RE.sub(" ", t)

    # normalize redup2 BEFORE punctuation stripping
    t = _apply_redup2(t, cfg)

    # normalize energy orthography variants (whitelist)
    t = normalize_energy_variants(t)

    # numbers
    if cfg.keep_numbers == "remove":
        t = _NUM_ONLY_RE.sub(" ", t)
    elif cfg.keep_numbers == "token":
        t = _NUM_ONLY_RE.sub(" <NUM> ", t)
    elif cfg.keep_numbers == "keep":
        pass
    else:
        raise ValueError("keep_numbers must be token|keep|remove")

    # punctuation stripping (toep hyphen)
    if cfg.remove_punct:
        t = _PUNCT_RE.sub(" ", t)

    # ===
    # 5) setelah tanda baca dibuang, sering tersisa "--" from "<--" (karena "<" hilang)
    #    ini yang kita bersihkan:
    t = DASH_RUN_RE.sub(" ", t)
    # ===

    # whitelist + whitespace
    t = _ALLOWED_UNSUP_RE.sub(" ", t)
    t = _WHITESPACE_RE.sub(" ", t).strip()

    # join whitelist bigrams into canonical totons (toeps phrases stable for UNSUP)
    t = apply_join_rules_text(t)
    return t


PUNCT_STRIP = ".,!?;:()[]{}\"'`"
def remove_stopwords(text: str, stopset: set) -> str:
    if not text or not stopset:
        return text
    out = []
    for tok in text.split():
        core = tok.strip(PUNCT_STRIP).lower()
        if not core:
            continue
        if core in stopset:
            continue
        out.append(tok)
    return " ".join(out)

## 5) Build stopset UNSUP vs ABSA_SAFE

- UNSUP: indo+eng + custom_unsup
- ABSA_SAFE: indo+eng + custom_absa, tetapi **negasi dipertahankan** (not diremove)

In [ ]:
NEGATIONS = {
    # id
    "tidak","tak","nggak","ngga","gak","ga","ndak","bukan","jangan","belum",
    # en
    "no","not","never","none","nothing","nope","cannot","can't","dont","don't","won't","without"
}

stop_unsup = indo_sw.union(eng_sw).union(stop_unsup_custom)

# ABSA-safe: buang negasi from stopset (walau ada di NLTK/custom)
stop_absa = indo_sw.union(eng_sw).union(stop_absa_custom)
stop_absa = set([w for w in stop_absa if w not in NEGATIONS])

print("stop_unsup size:", len(stop_unsup))
print("stop_absa size :", len(stop_absa))
print("negations kept :", sorted(list(NEGATIONS))[:10], "...")

## 6) Jalankan preprocessing for dataset + simpan output

In [ ]:
# =========================
# 6) Run preprocessing
# =========================
assert INPUT_CSV.exists(), f"Input file not found: {INPUT_CSV}"
df = pd.read_csv(INPUT_CSV)
assert TEXT_COL in df.columns, f"Column '{TEXT_COL}' not found. Available: {list(df.columns)[:30]}"

raw = df[TEXT_COL].fillna("").astype(str)

df_out = pd.DataFrame()
df_out["seg_text_raw"] = raw

# clean + slang normalize
df_out["unsup_processed"] = raw.apply(lambda x: clean_unsup(x, CFG)).apply(lambda x: normalize_slang(x, slang_map))

# UNSUP stopword filtering
df_out["unsup_filtered"] = df_out["unsup_processed"].apply(lambda x: remove_stopwords(x, stop_unsup))
# stemming removed (per request)
# ABSA-safe filtering (negation topt)
df_out["absa_filtered"] = df_out["unsup_processed"].apply(lambda x: remove_stopwords(x, stop_absa))

df_out.to_csv(OUT_CSV, index=False, encoding="utf-8")
print("[OK] Saved:", OUT_CSV, "| rows:", len(df_out))
df_out.head(8)

## 7) Audit: emoticon/emoji & redup2 before–after

In [ ]:
def has_kaskus_emo(text: str) -> bool:
    return bool(KASKUS_EMO_RE.search(str(text))) if text is not None else False

def has_emoji(text: str) -> bool:
    return bool(EMOJI_RE.search(str(text))) if text is not None else False

def has_redup2(text: str) -> bool:
    if text is None:
        return False
    # detect any word2
    return bool(re.search(r"\b[a-z]{3,}2\b", str(text).lower()))

def summarize_audit(df_out: pd.DataFrame) -> dict:
    raw = df_out["seg_text_raw"]
    proc = df_out["unsup_processed"]

    audit = {
        "rows": int(len(df_out)),
        "has_kaskus_emo_before": int(raw.apply(has_kaskus_emo).sum()),
        "has_kaskus_emo_after": int(proc.apply(has_kaskus_emo).sum()),
        "has_emoji_before": int(raw.apply(has_emoji).sum()),
        "has_emoji_after": int(proc.apply(has_emoji).sum()),
        "has_redup2_before": int(raw.apply(has_redup2).sum()),
        "has_redup2_after": int(proc.apply(has_redup2).sum()),
        "cfg": CFG.__dict__,
        "stopset_sizes": {"unsup": int(len(stop_unsup)), "absa": int(len(stop_absa))},
    }
    # --- energy orthography/join audit (whitelist) ---
    energy = {}
    for canon, pat in ENERGY_PATTERNS_FOR_AUDIT:
        try:
            before = int(raw.str.contains(pat, regex=True, na=False).sum())
        except Exception:
            before = None
        # after: count canonical toton occurrences in processed text
        try:
            after = int(proc.str.contains(rf"\b{re.escape(canon)}\b", regex=True, na=False).sum())
        except Exception:
            after = None
        # simple "fixed" heuristic: raw has variant pattern AND processed has canonical
        try:
            fixed = int((raw.str.contains(pat, regex=True, na=False) & proc.str.contains(rf"\b{re.escape(canon)}\b", regex=True, na=False)).sum())
        except Exception:
            fixed = None
        energy[canon] = {"rows_before": before, "rows_after": after, "rows_fixed": fixed, "pattern": pat}

    audit["energy_variants"] = energy
    audit["join_rules_2"] = {"n_rules": int(len(JOIN_RULES_2))}
    return audit

audit = summarize_audit(df_out)
AUDIT_JSON.write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
print("[OK] Saved:", AUDIT_JSON)
audit

In [ ]:
# audit samples: prioritaskan yang mengandung emoticon/emoji/redup2
samples = df_out.copy()
samples["has_kaskus_emo_before"] = samples["seg_text_raw"].apply(has_kaskus_emo)
samples["has_emoji_before"]      = samples["seg_text_raw"].apply(has_emoji)
samples["has_redup2_before"]     = samples["seg_text_raw"].apply(has_redup2)

priority = samples[(samples["has_kaskus_emo_before"]) | (samples["has_emoji_before"]) | (samples["has_redup2_before"])].head(80)
priority.to_csv(AUDIT_SAMPLES, index=False, encoding="utf-8")
print("[OK] Saved:", AUDIT_SAMPLES, "| rows:", len(priority))
priority[["seg_text_raw","unsup_processed","unsup_filtered","absa_filtered"]].head(12)

## 8) Quick tests 
- `:ngakak:` hilang total
- `negara2` jadi `negara-negara`
- `CO2` not dichange

In [ ]:
tests = [
    "Negara2 tropis bisa ... :toast",
    "Merusak tanah sekali, ckckck.",
    "Dikubur wkwkwk.",
    "CO2 naik, H2O turun, G20 meeting.",
    "tower2 listrik itu :ngakak: 😂😂",
]
for s in tests:
    print("BEFORE:", s)
    print("AFTER :", clean_unsup(s, CFG))
    print("-"*70)